In [ ]:
%%capture

from IPython.display import display, Javascript

!pip install evaluate
!pip install accelerate -U
!pip install sacrebleu

display(Javascript('google.colab.kernel.restart()'))

In [ ]:
import torch
from torch.nn import Transformer
from transformers import BertModel
import torch.nn as nn
from transformers import AdamW
from transformers import get_scheduler, TrainingArguments, Trainer
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
# import evaluate
import math
import json
import random
from sklearn.model_selection import train_test_split
from accelerate import Accelerator
import os
import nltk
from nltk.translate.bleu_score import corpus_bleu
import sacrebleu

In [ ]:
# Most models handle sequences of up to 512 or 1024 tokens,
# and will crash when asked to process longer sequences.
# if needed to handle: sequence = sequence[:max_sequence_length]

In [ ]:
# GLOBAL VARIABLES
device = 'cuda' if torch.cuda.is_available() else 'cpu'
target_seq_length = 10
min_seq_length = 6
max_seq_length = target_seq_length + min_seq_length

In [ ]:
class ChordTokenizer:
    def __init__(self, max_sequence_length):
        self.vocab = {'<pad>': 0}  # initialize with <pad> token
        self.idx_to_token = ['<pad>'] # this array is just for encoding/decoding ease
        self.max_sequence_length = max_sequence_length

    def create_vocab(self, sequences):
        for sequence in sequences:
            for chord in sequence['input_sequence'] + sequence['target_sequence']:
                if chord not in self.vocab:
                    self.vocab[chord] = len(self.vocab)
                    self.idx_to_token.append(chord)

    def tokenize_and_pad(self, sequence):
        max_length = self.max_sequence_length
        tokenized_sequence = [self.vocab.get(chord, self.vocab['<pad>']) for chord in sequence]
        padded_sequence = tokenized_sequence + [self.vocab['<pad>']] * (max_length - len(tokenized_sequence))
        return padded_sequence[:max_length]

    def decode_sequence(self, token_ids, skip_special_tokens=False):
        decoded_sequence = [self.idx_to_token[token_id] for token_id in token_ids]
        if skip_special_tokens:
            decoded_sequence = [token for token in decoded_sequence if token != '<pad>']
        return decoded_sequence

    def get_vocab_size(self):
        return len(self.vocab)

    def get_vocab(self):
        return self.vocab

    def get_idx_to_token(self):
        return self.idx_to_token

In [ ]:
class ChordDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Access the dictionary at index `idx`
        item = self.data[idx]

        # Extract the input_sequence and target_sequence
        input_sequence = item['input_sequence']
        target_sequence = item['target_sequence']

        # Convert to tensors and return
        return torch.tensor(input_sequence), torch.tensor(target_sequence)

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src = self.embedding(src)
        tgt = self.embedding(tgt)
        output = self.transformer(src, tgt)
        output = self.fc(output)
        return output

In [ ]:
# this takes the dataset (in the form of a 2d array) and returns it as a dictionary with input sequences and associated target sequences
def format_dataset(data):
    formatted_data = []
    for sequence in data:
        split_index = len(sequence)
        input_sequence = sequence[:split_index-1]
        target_sequence = sequence[1:split_index]
        formatted_data.append({'input_sequence': input_sequence, 'target_sequence': target_sequence})
    return formatted_data

In [ ]:
def split_sequence(seq, target_length=target_seq_length+1, min_length=min_seq_length):
    # we take the target sequence length and add 1 so we have an extra token for the target sequence (to predict)
    result = []
    # length of the current sequence
    n = len(seq)

    # compute the number of full chunks you would like
    num_full_chunks = n // target_length

    # compute the length of the last chunk
    last_chunk_length = n % target_length

    # split into full chunks
    for i in range(num_full_chunks):
        result.append(seq[i * target_length: (i + 1) * target_length])

    # handle the last chunk. if > len(15), append it, otherwise add (extend) it to previous array
    if last_chunk_length != 0:
        if last_chunk_length < min_length and len(result) > 0:
            # merge with the previous chunk
            result[-1].extend(seq[num_full_chunks * target_length:])
        else:
            # add as a separate chunk
            result.append(seq[num_full_chunks * target_length:])

    return result

In [ ]:
# open the json data file and read in the chords as a 2d array
with open('song_chords.json', 'r') as f:
    songs_data = json.load(f)

chord_sequences = []
for song in songs_data:
    chord_sequences.extend(split_sequence(song['chords']))

In [ ]:
print(f"Number of songs: {len(songs_data)}")
print(f"Total chord sequences (after splits): {len(chord_sequences)}")

Number of songs: 980
Total chord sequences (after splits): 8982


In [ ]:
# format the dataset with input and target sequences, create vocab, and tokenize
chord_dataset = format_dataset(chord_sequences)

tokenizer = ChordTokenizer(max_sequence_length=max_seq_length)
tokenizer.create_vocab(chord_dataset)

tokenized_dataset = []
for chords in chord_dataset:
    input_sequence = tokenizer.tokenize_and_pad(chords['input_sequence'])
    target_sequence = tokenizer.tokenize_and_pad(chords['target_sequence'])
    tokenized_dataset.append({'input_sequence': input_sequence, 'target_sequence': target_sequence})

print(tokenized_dataset[0])

{'input_sequence': [1, 2, 3, 4, 1, 2, 1, 1, 5, 3, 0, 0, 0, 0, 0, 0], 'target_sequence': [2, 3, 4, 1, 2, 1, 1, 5, 3, 4, 0, 0, 0, 0, 0, 0]}


In [ ]:
chord_sequences_train, chord_sequences_validation = train_test_split(tokenized_dataset, test_size=0.2, random_state=42)

In [ ]:
train_dataset = ChordDataset(chord_sequences_train)
val_dataset = ChordDataset(chord_sequences_validation)

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32)
val_dataloader = DataLoader(val_dataset, batch_size=32)

In [ ]:
# for batch_idx, (src, tgt) in enumerate(train_dataloader):
#     print(f'Batch {batch_idx}:')
#     print(f'Source Sequence (input): {src}')
#     print(f'Target Sequence (output): {tgt}')
#     print('-----------------------')

#     break

In [ ]:
for batch_idx, (src, tgt) in enumerate(train_dataloader):
    print(f'Batch {batch_idx}:')
    print(f'Source Sequence (input) size: {src.size()}')
    print(f'Target Sequence (output) size: {tgt.size()}')

    # Check sequence lengths within the batch
    assert src.size(1) == tgt.size(1), f"Sequence length mismatch in batch {batch_idx}: src={src.size(1)}, tgt={tgt.size(1)}"

    # Print example content of src and tgt for further inspection
    print('Example Source Sequence:', src[0])
    print('Example Target Sequence:', tgt[0])

    print('-----------------------')

    break

Batch 0:
Source Sequence (input) size: torch.Size([32, 16])
Target Sequence (output) size: torch.Size([32, 16])
Example Source Sequence: tensor([ 51, 229, 207,   6, 318,  51, 229, 207,   6,  22, 319, 131,   6, 320,
        312,   0])
Example Target Sequence: tensor([229, 207,   6, 318,  51, 229, 207,   6,  22, 319, 131,   6, 320, 312,
          5,   0])
-----------------------


In [ ]:
# Initialize model and move to GPU if available
model = TransformerModel(len(tokenizer.vocab))
model.to(device)

# Define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training and validation loop
num_epochs = 10
generated_seqs = []
target_seqs = []

for epoch in range(num_epochs):
    # Training loop
    model.train()
    epoch_train_loss = 0
    for src, tgt in tqdm(train_dataloader):
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()


        output = model(src, tgt)
        loss = criterion(output.view(-1, output.size(-1)), tgt.contiguous().view(-1))
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    # Validation loop
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for src_val, tgt_val in val_dataloader:
            src_val, tgt_val = src_val.to(device), tgt_val.to(device)
            output_val = model(src_val, tgt_val)
            loss_val = criterion(output_val.view(-1, output_val.size(-1)), tgt_val.contiguous().view(-1))
            epoch_val_loss += loss_val.item()

            generated_tokens = torch.argmax(output_val, dim=-1)
            for i in range(len(generated_tokens)):
                generated_seq = tokenizer.decode_sequence(generated_tokens[i].tolist(), skip_special_tokens=True)
                target_seq = tokenizer.decode_sequence(tgt_val[i].tolist(), skip_special_tokens=True)
                print(f"Prediction: {' '.join(generated_seq)}")
                print(f"Target: {' '.join(target_seq)}")


            # generated_tokens = torch.argmax(output_val, dim=-1)
            # for seq in generated_tokens.tolist():
            #     decoded_seq = tokenizer.decode_sequence(seq, skip_special_tokens=True)
            #     generated_seqs.append(decoded_seq)

            # for seq in tgt_val.tolist():
            #     decoded_seq = tokenizer.decode_sequence(seq, skip_special_tokens=True)
            #     target_seqs.append(decoded_seq)


    # print(f'generated_seqs len: {len(generated_seqs)}, ex: {generated_seqs[0]}')
    # print(f'target_sequences len: {len(target_seqs)}, ex: {target_seqs[0]}')
    # bleu_score_nltk = corpus_bleu(target_seqs, generated_seqs)
    # bleu_score_sacrebleu = sacrebleu.corpus_bleu(generated_seqs, target_seqs)
    # print(f'BLEU Score (NLTK): {bleu_score_nltk:.4f}')
    # print(f'BLEU Score (SacreBLEU): {bleu_score_sacrebleu.score:.4f}')

    # Print epoch statistics
    print(f'Epoch {epoch+1}/{num_epochs}, '
          f'Train Loss: {epoch_train_loss/len(train_dataloader):.4f}, '
          f'Val Loss: {epoch_val_loss/len(val_dataloader):.4f}')


In [ ]:
# Unwrap the model from the accelerator
unwrapped_model = accelerator.unwrap_model(model)

# Save the model
save_directory = "saved_models"
os.makedirs(save_directory, exist_ok=True)
model_save_path = os.path.join(save_directory, "chords_model.pth")

torch.save(unwrapped_model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
%%capture
model.load_state_dict(torch.load(model_save_path))
model = accelerator.prepare(model)
model.eval()

In [ ]:
model.to(device)

In [ ]:
input_chords = ["G", "C", "D", "G", "C", "D", "F", "G", "F"]  # Example input chord sequence
predicted_chords = predict_chords(model, tokenizer, input_chords)
print("Input Chords:", input_chords)
print("Predicted Chords:", predicted_chords)

In [ ]:
def predict_chords(model, tokenizer, input_chords, max_length=512):
    # encode the input chords
    encoded_chords = tokenizer.encode(input_chords, max_length=max_length)

    input_ids = torch.tensor([encoded_chords], dtype=torch.long)
    attention_mask = torch.ones_like(input_ids)

    input_ids = accelerator.prepare(input_ids)
    attention_mask = accelerator.prepare(attention_mask)

    input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)

    # run the model
    with torch.no_grad():
        _, logits = model(input_ids=input_ids, attention_mask=attention_mask)

    # get the predicted token indices
    preds = torch.argmax(logits, dim=-1).squeeze().tolist()

    # decode the token indices back to chords
    predicted_chords = []
    for idx in preds:
        token = tokenizer.decode([idx])[0]  # Decode single index
        if token != '<pad>':  # Ignore padding tokens
          predicted_chords.append(token)

    # Join the predicted chords into a single string
    predicted_chords = ' '.join(predicted_chords)

    return predicted_chords